## Task 2
### Integrantes
* Sergio Orellana 221122
* Rodrigo Mansilla 22611
* Ricardo Chuy 221007

## Pregunta 2.1

### Pregunta 1

No estoy de acuerdo con el desarrollador. Aunque ImageNet contiene fotos naturales y las radiografías sean imágenes en escala de grises, las capas convolucionales tempranas no aprenden "perros" o "autos"; aprenden primitivas visuales generales como bordes, contornos, gradientes, orientaciones y texturas. Esas estructuras También existen en radiografías, por ejemplo en limites costales, opacidades, patrones pulmonares y transiciones de intensidad. Por eso considero que ese conocimiento si es transferible entre dominios visualmente distintos.

Desde mi perspectiva, la diferencia entre dominios afecta más a las capas profundas que a las primeras. Las capas iniciales capturan patrones de bajo nivel bastante universales; en cambio, las capas altas son las que se vuelven más específicas al dataset original. En consecuencia, usar Transfer Learning no "contamina" el modelo: me da un punto de partida mucho mejor que entrenar todos los filtros desde cero con solo 800 imágenes.


### Pregunta 2

más allá del costo computacional, entrenar desde cero me cuesta tiempo al mercado. Si parto sin pesos preentrenados, necesito más iteraciones, más experimentos fallidos y más tiempo para estabilizar el entrenamiento. En una startup médica, ese retraso significa tardar más en validar, presentar resultados y llegar a una version útil para clínicas rurales.

También me cuesta riesgo tecnico y humano. Con pocos datos, la probabilidad de terminar con un modelo fragil o sobreajustado es mayor; por tanto, el riesgo de tomar malas decisiones de producto también sube. Además, entrenar desde cero exige más experiencia en optimización, regularización y diagnóstico de entrenamiento. Eso implica depender más de talento especializado, que suele ser escaso y costoso.


### Pregunta 3

Si consideraría entrenar desde cero en un escenario donde disponga de un dataset grande, representativo y propio del dominio médico, por ejemplo decenas o cientos de miles de radiografías bien etiquetadas y provenientes de equipos similares a los de despliegue. También lo evaluaría si la modalidad de imagen fuera muy distinta a la estadistica visual que aprovecha el preentrenamiento comun, o si el objetivo clinico requiriera una arquitectura diseñada específicamente para ese dominio.

En ese caso, entrenar desde cero tendría sentido porque ya no dependería tanto del conocimiento prestado de otro dataset. Sin embargo, con 800 imágenes yo no lo defendería como primera opcion.


## Pregunta 2.2

### Pregunta 4

Dado que solo tengo 800 imágenes y el dominio es distinto a ImageNet, yo recomendaría iniciar con la Opcion A: congelar toda la red base y entrenar unicamente el cabezal final. La logica no es repetir una regla de oro, sino administrar riesgo. Con tan pocos datos, si descongelo toda la red desde el inicio, expongo millones de parametros a ajustarse con evidencia insuficiente. Eso aumenta la probabilidad de sobreajuste y de destruir representaciones utiles que el backbone ya había aprendido.

En cambio, al congelar la base uso el extractor preentrenado como una fuente estable de caracteristicas y concentro el aprendizaje en la parte más específica para la tarea binaria. Asi obtengo una linea base sólida, más rapida y menos riesgosa. Si luego observo que el cabezal se queda corto, recien consideraría un fine-tuning parcial y controlado.


### Pregunta 5

Si aplico la Opcion B con una tasa alta como $1e-3$ en toda la red, corro el riesgo de destruir rapidamente la informacion util del preentrenamiento. Ese fenomeno se conoce como catastrophic forgetting. En lugar de adaptar el modelo con cuidado, haría actualizaciones demasiado agresivas sobre todos los pesos, incluyendo filtros tempranos que todavía son valiosos.

Durante el entrenamiento yo esperaría ver inestabilidad: oscilaciones fuertes en la loss, caidas bruscas en rendimiento de validacion, o incluso un comportamiento donde el modelo no converge bien. En otras palabras, el modelo no estaría refinando conocimiento, sino sobrescribiendolo demasiado rapido.


### Pregunta 6 

Si MediScan Guatemala llegara a tener 50000 radiografias etiquetadas, si cambiaría mi recomendacion. Con ese volumen de datos ya tendría evidencia suficiente para descongelar progresivamente más capas y hacer un fine-tuning más profundo. Primero mantendría una etapa de entrenamiento del cabezal; despues, descongelaría los últimos bloques del backbone con una tasa pequeña, por ejemplo $1e-5$ o $1e-4$ segun la estabilidad observada.

La razon es que, con más datos, el riesgo de sobreajuste baja y el beneficio de adaptar mejor las capas profundas al dominio médico aumenta. En ese escenario, una estrategia gradual me permitiría capturar señales más específicas de radiografias sin perder de golpe las ventajas del preentrenamiento.


## Pregunta 2.3

### Pregunta 7

Si considero viable reutilizar parte del modelo, pero no lo reutilizaría directamente sin modificaciones. Las primeras capas probablemente siguen siendo utiles, porque ya aprendieron a detectar bordes, contornos, transiciones de intensidad y patrones basicos que También existen en radiografias de muneca. Sin embargo, las capas más profundas y el cabezal final estan más alineados con el problema de neumonía en torax, no con fracturas óseas en otra anatomia.

Por eso, yo aprovecharía el backbone como punto de partida, pero cambiaría el cabezal de clasificacion y revisaría que tanto conviene reajustar bloques superiores. 


### Pregunta 8

Paso 1. Reemplazaría el cabezal final por uno nuevo adaptado a la tarea de fractura de muneca. Inicialmente congelaría toda la base y entrenaría solo ese cabezal con una tasa relativamente moderada, por ejemplo $1e-3$, para obtener una linea base estable.

Paso 2. Si el desempeño inicial es razonable pero insuficiente, descongelaría solo los últimos bloques del backbone y mantendría una tasa mucho más baja en esa parte, por ejemplo $1e-5$, mientras dejo el cabezal con una tasa algo mayor. Asi permito adaptacion específica sin sobrescribir por completo el conocimiento previo.

Paso 3. Validaría el modelo en un conjunto realmente representativo del nuevo dominio, idealmente separado por paciente y por fuente hospitalaria. Si detecto sesgos o mala generalizacion, reajustaría el grado de descongelamiento y la estrategia de entrenamiento antes de pensar en despliegue.


### Pregunta 9

El principal riesgo etico y clinico es asumir que un modelo entrenado para una patología y una anatomía puede generalizar automaticamente a otra tarea medica. Si hago eso sin validacion rigurosa, puedo inducir errores diagnosticos, generar falsa confianza en el personal clinico y perjudicar pacientes. En medicina, un error no es solo una metrica mala: puede convertirse en retraso de tratamiento o en decisiones clinicas equivocadas.

Antes de desplegarlo, yo recomendaría como minimo una validacion independiente con datos del nuevo dominio, revision por expertos clinicos, analisis de sensibilidad, especificidad y errores por subgrupos, y una prueba piloto controlada. Solo despues de demostrar desempeño consistente y clinicamente aceptable consideraría llevarlo a produccion.


## Referencias

- Contributors, P. (2023, January 1). Transfer Learning for Computer Vision tutorial. https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html 
- GeeksforGeeks. (2024, March 20). ML | Transfer Learning with Convolutional Neural Networks. GeeksforGeeks. https://www.geeksforgeeks.org/deep-learning/ml-transfer-learning-with-convolutional-neural-networks/ 
- GeeksforGeeks. (2025a, July 23). Transfer learning & finetuning using Keras. GeeksforGeeks. https://www.geeksforgeeks.org/deep-learning/transfer-learning-fine-tuning-using-keras/ 
- GeeksforGeeks. (2025b, August 6). Deep Transfer Learning Introduction. GeeksforGeeks. https://www.geeksforgeeks.org/deep-learning/deep-transfer-learning-introduction/ 
- GeeksforGeeks. (2025c, December 17). Difference between FineTuning and transfer learning. GeeksforGeeks. https://www.geeksforgeeks.org/machine-learning/what-is-the-difference-between-fine-tuning-and-transfer-learning/

